Order of work
This follows the spec's suggested order (5.2). Nobody merges into the integration branch unless make check in src/vm still passes everything that passed before. Set a date for each phase in the group chat; the Phase 3 date is the trigger for D's backup plan.
1.  Phase 0, before anything: Task 2 is merged into master with all 86 tests passing. Create task3-integration from master. Each person branches from it: task3-frames (A), task3-pages (B), task3-mmap (C), task3-stack (D).
2.  Phase 1, day one (A and B, everyone reads it): the four headers, the stub .c files, the Makefile.build lines and the new struct process fields. Gate: it builds in src/vm, and every Task 2 test still passes there.
3.  Phase 2 (A then B): A's frame_alloc with no eviction (it returns NULL when palloc fails), then B's page table, fault handler, lazy load_segment and setup_stack through frames. Gate: all Task 2 functionality tests, pt-bad-addr, pt-bad-read, pt-write-code and pt-write-code2 pass.
4.  Phase 3, in parallel: C does mmap, munmap and syscall pinning. D does stack growth. A does sharing. B does page_table_destroy and the process_exit order. Gate: every mmap test except page-merge-mm, every pt-grow test, and multi-oom (which catches leaks).
5.  Phase 4 (A and B, C helps with pinning): eviction and swap. Gate: page-linear, page-parallel, page-shuffle, the four page-merge tests and pt-overflowstk pass. The full suite passes three runs in a row, since eviction bugs are often timing dependent.
6.  Phase 5 (everyone): a style pass on your own code (see Rules for everyone). Read every other person's code until you can explain it, because the code review accepts no "I didn't write that". D finishes doc/vm.md. Merge task3-integration into master and check LabTS.
Mark sheet and style points
The Task 3 code review is marked out of 18. Two rows are hidden under comment icons on the sheet (VM race conditions and read/write buffers) and share the remaining 8 marks. Race conditions has the biggest box, so treat it as the single most valuable row.
Marks
Review point
Owner
What earns it
2
Read-only executable sharing
A, B calls it
Share table keyed by (inode, page number), a frame with an owner list, accessed and dirty bits read from every owner's PTE
1
Stack fault heuristic and growth limit
D
stack_is_access: no more than 32 bytes below esp, above STACK_BOTTOM; 8 MB limit
1
Advanced eviction algorithm
A
Enhanced second chance on the accessed and dirty bits
8, with the buffers row
Avoids VM race conditions
A, B, C
Synchronisation rules: page lock before frame lock, try-acquire in eviction, PTE cleared before saving
1
Avoids page-fault I/O contention
B
File and swap I/O hold only the page lock, never frame_lock
(in the 8)
Safe handling of read/write syscall buffers
C
Pin one page at a time around file_read and file_write; never fault holding the file system lock
1
Efficient use of swap slots
B, A
Only dirty pages go to swap, slots taken only at eviction, freed by swap_in, munmap and exit
1
Minimal file-system communication
B, C
Read exactly read_bytes once per load; zero pages never read; clean pages never written back; shared pages read once for everyone; file_read_at with no seeks; one file_reopen per mapping, not per page
3
No memory leaks on process termination
B, C, A
mmap_unmap_all, page_table_destroy (frames, swap slots, struct page), frame_release frees the struct frame and its share entry; multi-oom passes
The style points come from the Meets Expectations list. The seven in red on the sheet are marked (red); do those first.
Style point
Owner
How
Function level code commenting (red)
Everyone
A comment above every function: what it does, what it returns, which locks the caller must hold
Lazy loading of exec/mmap pages (red)
B, C
load_segment and mmap_map only record pages; data is read in page_load
Safe hash_less comparisons (red)
A, B
Compare with <, never by subtracting (it overflows); use the same key as the hash; share table compares the inode with file_less, then the offset
Synchronisation of frame eviction (red)
A
frame_lock for the choice only, try-acquire page locks, evicting flag, I/O outside frame_lock
Sensible eviction algorithm (red)
A
Enhanced second chance that skips pinned frames
SEL_KCSEG page-faults still PANIC (red)
B
page_fault handles only user-mode faults; kernel faults still reach kill and PANIC
Locality of mapid and mmap table (red)
C
struct mmap_table holds the list and the next id; only vm/mmap.c touches it
Phase level code commenting
Everyone
A short comment before each stage of long functions (page_load, frame_evict, mmap_map), never restating a line
Consistent code layout
Everyone
Skeleton GNU style, braces always, under 80 columns
Function naming conventions
Everyone
Module prefixes: frame_, page_, mmap_, stack_
Efficient read-only sharing look-up
A
One hash lookup by (inode, page number), never a list scan
Obvious hash collisions avoided
A, B
Hash the page number, not raw addresses or offsets whose low 12 bits are always zero
Sharing implemented for mmap file pages
A, C (stretch)
Only after Phase 4 passes: clean mmap pages of the same file and offset share a frame, like code pages
Eviction fairness
A
One global clock hand that keeps moving, so no process's pages are always picked first
Efficient read/write buffer size
C
Page-sized chunks, no kernel copy of the whole user buffer
Efficient mmap overlap check
C
One page_lookup per page in the range before creating anything, stopping at the first hit
No magic numbers
Everyone
STACK_MAX, PUSHA_REACH, MAP_FAILED, PGSIZE; never 4096 or 32 in code
No superfluous locking
Everyone
No lock on data only its own thread touches: the spt hash and the mmap table belong to one process
No additional performance issues, no inefficient memory allocations
Everyone
No byte-by-byte loops over pages, no malloc while evicting, one struct page per page
Return status of kernel calls checked
Everyone
malloc, palloc_get_page, hash_init, hash_insert, file_read_at, file_reopen, pagedir_set_page, swap_out
Git use, commit quality, conventional commits
Everyone
Rules for everyone below
Rules for everyone
Task 1 lost most of its marks on style and git (Overall Project Quality 25%), so these apply to every commit.
1.  One git identity per person. Run git config --global user.name and git config --global user.email with the same values on every machine. The Task 1 marker named two of us for this and warned of penalties.
2.  Conventional commits with a scope: feat(vm): ..., fix(vm): ..., refactor(userprog): ..., ending with your login in brackets. Keep the subject under 72 characters. Never blame a teammate in a message.
3.  Small commits, often. One logical change each. Markers check commit frequency and quality.
4.  Braces on every if, for and while body, even single statements. This was in the Task 1 feedback.
5.  A comment above every new function saying what it does and what it returns. No comments that just restate the line below them.
6.  No magic numbers. Name them as the headers do: STACK_MAX, PUSHA_REACH, MAP_FAILED.
7.  Check the return value of every kernel call that can fail: malloc, palloc_get_page, hash_init, hash_insert, file_read_at, file_reopen, pagedir_set_page.
8.  Never run a formatter over a whole file, and don't reformat code you didn't change. In Task 2 that caused merge conflicts for everyone.
9.  Before every commit, run git diff --check (no trailing spaces) and keep lines under 80 columns.
10.  Never commit test hacks, scratch handlers or debug printfs.
11.  Before every merge request, run make check in src/vm and paste the FAIL lines into the request.
12.  Know everyone's code. The code review is in person, and "someone else wrote that" is not accepted.